# Notebook 03 - TF-IDF và K-Means phân cụm chủ đề

Notebook này biểu diễn bình luận bằng TF-IDF, chọn số cụm K và phân tích xem cụm K-Means có tương đồng với nhãn aspect thực tế hay không.

## 0. Khởi tạo môi trường

In [ ]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = Path("results/figures")
CSV_DIR = Path("results/csv")
MODEL_DIR = Path("results/models")
for path in [FIG_DIR, CSV_DIR, MODEL_DIR, Path("datasets/cleaned")]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

## 3.1 Xây dựng TF-IDF

TF-IDF đo mức quan trọng của từ trong từng bình luận so với toàn bộ corpus. `sublinear_tf=True` dùng log-scaling để giảm ảnh hưởng của từ lặp lại quá nhiều trong một bình luận.

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from src.clustering import (
    build_tfidf,
    compare_cluster_vs_aspect,
    find_optimal_k,
    get_top_keywords,
    plot_pca_clusters,
    plot_wordclouds,
    run_kmeans,
)

train_clean = pd.read_csv("datasets/cleaned/train_clean.csv", encoding="utf-8-sig")
vectorizer, X_tfidf = build_tfidf(
    train_clean["comment"], max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
)
print("Shape ma trận TF-IDF:", X_tfidf.shape)
print(f"Nhận xét: Có {X_tfidf.shape[0]} bình luận và {X_tfidf.shape[1]} đặc trưng từ/ngữ.")

row = X_tfidf[0].toarray().ravel()
terms = vectorizer.get_feature_names_out()
top_idx = row.argsort()[-15:][::-1]
sample_tfidf = pd.DataFrame({"term": terms[top_idx], "tfidf": row[top_idx]})
display(sample_tfidf)

plt.figure(figsize=(8, 5))
sns.barplot(data=sample_tfidf, x="tfidf", y="term", palette="mako")
plt.title("Top TF-IDF của một bình luận mẫu")
plt.tight_layout()
plt.savefig(FIG_DIR / "tfidf_sample_top_terms.png", dpi=150)
plt.show()

## 3.2 Tìm K tối ưu

In [ ]:
k_scores = find_optimal_k(X_tfidf, k_range=range(2, 11), save_path=FIG_DIR)
display(k_scores.round(4))

best_row = k_scores.sort_values("silhouette", ascending=False).iloc[0]
best_k = int(best_row["k"])
print(f"K được chọn: {best_k}")
print(f"Lý do: K={best_k} có Silhouette Score cao nhất trong khoảng thử nghiệm ({best_row['silhouette']:.4f}).")
print("Nhận xét: Nếu Elbow không có điểm gãy rõ, Silhouette là tiêu chí định lượng giúp chọn K ổn định hơn cho dữ liệu văn bản thưa.")

## 3.3 Chạy K-Means với K tối ưu

In [ ]:
kmeans_model, labels, final_silhouette = run_kmeans(X_tfidf, best_k, random_state=42)
train_clean["cluster"] = labels

cluster_counts = train_clean["cluster"].value_counts().sort_index()
display(cluster_counts.to_frame("count"))
print(f"Silhouette Score cuối: {final_silhouette:.4f}")

largest_pct = cluster_counts.max() / cluster_counts.sum() * 100
if largest_pct > 60:
    print(f"Cảnh báo: Cụm lớn nhất chiếm {largest_pct:.2f}% tổng dữ liệu. Điều này thường xảy ra khi TF-IDF tạo không gian thưa.")
else:
    print(f"Nhận xét: Cụm lớn nhất chiếm {largest_pct:.2f}%, chưa vượt ngưỡng lệch 60%.")

plt.figure(figsize=(8, 5))
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette="Set3")
plt.title("Số lượng bình luận mỗi cụm K-Means")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_cluster_size.png", dpi=150)
plt.show()

## 3.4 Phân tích top keywords mỗi cụm

In [ ]:
top_keywords = get_top_keywords(kmeans_model, vectorizer, n_top=10)
for cluster_id, keywords in top_keywords.items():
    display(Markdown(f"### Cụm {cluster_id}"))
    print("Top 10 keywords:", ", ".join(keywords))
    examples = train_clean.loc[train_clean["cluster"] == cluster_id, "comment"].head(3).tolist()
    print("Ví dụ bình luận thực tế trong cụm:")
    for i, comment in enumerate(examples, start=1):
        print(f"{i}. {comment}")
    print("Nhận xét: Dựa trên keywords và ví dụ, có thể suy luận chủ đề chính của cụm trước khi so sánh với aspect thực tế.\n")

## 3.5 So sánh cụm K-Means với aspect label thực tế

In [ ]:
crosstab, cluster_summary = compare_cluster_vs_aspect(train_clean, "cluster", "main_aspect")
display(crosstab)
display(cluster_summary.assign(match_ratio_pct=lambda x: (x["match_ratio"] * 100).round(2)))

cluster_names = dict(zip(cluster_summary["cluster"], cluster_summary["cluster_name"]))
train_clean["cluster_name"] = train_clean["cluster"].map(cluster_names)
good_clusters = (cluster_summary["match_ratio"] >= 0.5).sum()
print(f"Số cụm khớp tốt (match_ratio >= 50%): {good_clusters}/{len(cluster_summary)}")
print("Nhận xét: Cụm có match_ratio cao cho thấy K-Means tìm được chủ đề gần với aspect thực tế; cụm thấp thường trộn nhiều chủ đề.")

## 3.6 Visualize và lưu kết quả

In [ ]:
plot_pca_clusters(X_tfidf, labels, cluster_names, FIG_DIR / "pca_kmeans.png")
plot_wordclouds(train_clean, "cluster", "comment", FIG_DIR)

train_clean.to_csv(CSV_DIR / "kmeans_clustered.csv", index=False, encoding="utf-8-sig")
joblib.dump(kmeans_model, MODEL_DIR / "kmeans_model.pkl")
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.pkl")
cluster_summary.to_csv(CSV_DIR / "kmeans_cluster_summary.csv", index=False, encoding="utf-8-sig")
k_scores.to_csv(CSV_DIR / "kmeans_k_scores.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả phân cụm, model và hình trực quan.")